Build the static PPI graph based on the cleaned dataset -> 1060 proteins 

Load STRING PPI data

In [56]:
import sys
import pandas

print("Python:", sys.executable)
print("Pandas version:", pandas.__version__)

Python: /home/vascul/vsayyalasomayajula/my-scratch/miniconda3/envs/GNN/bin/python
Pandas version: 2.2.3


In [57]:
import pandas as pd
import torch
from torch_geometric.data import Data

In [58]:
import pandas as pd
import os
os.chdir('/home/vascul/vsayyalasomayajula/my-rdisk/r-divb/venkat/Proteomics/')
ppi_data = pd.read_csv('./PPI_STRING/interactions_with_HGNC_symbols.csv')

In [59]:
import torch
print(torch.version.cuda)


12.4


In [60]:
protein_list_df = pd.read_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/expression_df_imputed_RF_QRILC.csv')
protein_list_df.head(10)
#protein_list_df.iloc[0,0]
proteins = protein_list_df.iloc[:, 0].tolist()
node_index_df = pd.DataFrame({
    'node_index': range(len(proteins)),  
    'Gene': proteins 
})


node_index_df.to_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/original_node_indices_RF_QRILC.csv', index=False)

print(node_index_df.head(10))

   node_index   Gene
0           0  MAPK1
1           1  SUGT1
2           2   RBX1
3           3   PEPD
4           4  EEF1G
5           5   VTA1
6           6   LSM5
7           7  PFDN1
8           8  PSMD5
9           9    RAN


In [61]:
print(proteins[:5])

['MAPK1', 'SUGT1', 'RBX1', 'PEPD', 'EEF1G']


In [62]:
included_proteins = protein_list_df['proteins']
DEP = pd.read_csv('./PlasmaAAA/DEP.csv', header=None)
DEP = DEP.drop_duplicates()

common_proteins = DEP[DEP[0].isin(included_proteins)]
print(len(common_proteins))

396


Construct a graph based on variational interaction strength .. for DEP proteins 100, for all else 500

In [76]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data

protein_to_idx = {protein: idx for idx, protein in enumerate(proteins)}

DEP = pd.read_csv('./PlasmaAAA/DEP.csv', header=None)
DEP = DEP.drop_duplicates()
common_proteins = set(DEP[DEP[0].isin(included_proteins)][0])
print(len(common_proteins))

def get_threshold(row):
    both_in_DEP = (row['protein1'] in common_proteins) or (row['protein2'] in common_proteins)
    return 100 if both_in_DEP else 500

filtered_ppi_data = ppi_data.copy()
filtered_ppi_data['threshold'] = filtered_ppi_data.apply(get_threshold, axis=1)
filtered_ppi_data = filtered_ppi_data[filtered_ppi_data['combined_score'] >= filtered_ppi_data['threshold']]

sorted_pairs = np.sort(filtered_ppi_data[['protein1', 'protein2']], axis=1)
filtered_ppi_data = filtered_ppi_data.assign(
    p1=sorted_pairs[:, 0],
    p2=sorted_pairs[:, 1]
)

filtered_ppi_data = filtered_ppi_data.sort_values('combined_score', ascending=False)\
                                     .drop_duplicates(['p1', 'p2'])

edges = filtered_ppi_data[['p1', 'p2']].applymap(lambda protein: protein_to_idx.get(protein))
valid_edges = edges.dropna().astype(int)

edge_index = torch.tensor(valid_edges.to_numpy().T, dtype=torch.long)
edge_attr = torch.tensor(filtered_ppi_data.loc[valid_edges.index, 'combined_score'].values, dtype=torch.float)
num_nodes = len(protein_to_idx)
x = torch.ones((num_nodes, 1))
print(edge_index.shape, edge_attr.shape)

396


/home/vascul/vsayyalasomayajula/tmp/ipykernel_3596871/3721412678.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  edges = filtered_ppi_data[['p1', 'p2']].applymap(lambda protein: protein_to_idx.get(protein))


torch.Size([2, 85916]) torch.Size([85916])


In [77]:
print(x.shape)

torch.Size([1406, 1])


In [79]:
import torch
import numpy as np

edges_set = set()

for i in range(edge_index.shape[1]):
    node1, node2 = edge_index[:, i].tolist()
    if (node1, node2) not in edges_set and (node2, node1) not in edges_set:
        edges_set.add((node1, node2))

# Print results
print("Edge pairs (with both directions considered):")
print(len(edges_set))

# Check if both directions exist for each pair in the edge_index
both_directions = True
for i in range(edge_index.shape[1]):
    node1, node2 = edge_index[:, i].tolist()
    if (node1, node2) not in edges_set and (node2, node1) not in edges_set:
        both_directions = False
        break

print(f"Contains both directions: {both_directions}")


Edge pairs (with both directions considered):
85916
Contains both directions: True


Create a pytorch geometric data object

In [80]:
import torch
import torch_geometric.utils as pyg_utils
from torch_geometric.data import Data

edge_index, edge_attr = pyg_utils.to_undirected(edge_index, edge_attr=edge_attr)

data = Data(x=x,  
            edge_index=edge_index, 
            edge_attr=edge_attr)

print(data)

Data(x=[1406, 1], edge_index=[2, 171832], edge_attr=[171832])


In [81]:
data.is_undirected()

True

In [82]:
unique_edges = set(tuple(edge) for edge in edge_index.T.numpy())
undirected_check = all([tuple(reversed(edge)) in unique_edges for edge in unique_edges])

unique_edges_check = len(unique_edges) == edge_index.shape[1]

print("Undirected check passed:", undirected_check)
print("Unique edges check passed:", unique_edges_check)

Undirected check passed: True
Unique edges check passed: True


In [83]:

unique_nodes = torch.unique(edge_index)
print(f"Unique nodes in edge_index: {len(unique_nodes)}")

import pandas as pd

edges_df = pd.DataFrame(edge_index.T.numpy(), columns=['src', 'dst'])

edges_df_normalized = edges_df.apply(lambda row: sorted([row['src'], row['dst']]), axis=1, result_type='expand')
edges_df_normalized.columns = ['node1', 'node2']

num_unique_edges = edges_df_normalized.drop_duplicates().shape[0]

print(f"Unique edges: {num_unique_edges}")
print(f"Total edges: {edges_df.shape[0]}")



Unique nodes in edge_index: 1406
Unique edges: 85916
Total edges: 171832


Check which proteins have no edges

In [84]:
all_protein_indices = set(protein_to_idx.values())

nodes_with_edges = set(edge_index[0].tolist()).union(set(edge_index[1].tolist()))

proteins_without_edges = all_protein_indices - nodes_with_edges

proteins_without_edges_names = [protein for protein, idx in protein_to_idx.items() if idx in proteins_without_edges]

missing_proteins_df = pd.DataFrame(proteins_without_edges_names, columns=["missing proteins"])
missing_proteins_df.to_csv("missing_proteins.csv", index=False)

print(f"Proteins without edges: {proteins_without_edges_names}")
print(f"Number of proteins without edges: {len(proteins_without_edges_names)}")

Proteins without edges: []
Number of proteins without edges: 0


In [85]:
missing_proteins_set = set(proteins_without_edges_names)
intersection = missing_proteins_set.intersection(common_proteins)
print("Intersection of missing proteins and common proteins:", intersection)
print(f"Number of missing proteins that are in common proteins: {len(intersection)}")

Intersection of missing proteins and common proteins: set()
Number of missing proteins that are in common proteins: 0


Create a new index map for the graph component with edges


In [86]:
import torch
import pandas as pd

protein_to_idx = {protein: idx for idx, protein in enumerate(proteins)}

all_protein_indices = set(protein_to_idx.values())
nodes_with_edges = set(edge_index[0].tolist()).union(set(edge_index[1].tolist()))

proteins_without_edges = all_protein_indices - nodes_with_edges


proteins_with_edges = [protein for protein, idx in protein_to_idx.items() if idx not in proteins_without_edges]

new_protein_to_idx = {protein: idx for idx, protein in enumerate(proteins_with_edges)}

new_edge_index = []
for i, edge in enumerate(edge_index.t().tolist()): 
    protein1_idx, protein2_idx = edge
    if protein1_idx not in proteins_without_edges and protein2_idx not in proteins_without_edges:
        new_edge_index.append([new_protein_to_idx[proteins[protein1_idx]], new_protein_to_idx[proteins[protein2_idx]]])

# Convert to PyTorch tensor
new_edge_index = torch.tensor(new_edge_index, dtype=torch.long).t().contiguous()

# Step 6: Create a reverse mapping for the new indices to proteins (after reset)
new_idx_to_protein = {idx: protein for protein, idx in new_protein_to_idx.items()}

# Step 7: Save the missing proteins to a CSV file
missing_proteins = [proteins[i] for i in proteins_without_edges]
missing_proteins_df = pd.DataFrame(missing_proteins, columns=["missing proteins"])
missing_proteins_df.to_csv("missing_proteins.csv", index=False)

# Print results
print(f"Total proteins: {len(proteins)}")
print(f"Proteins without edges: {len(proteins_without_edges)}")
print(f"Proteins with edges: {len(proteins_with_edges)}")
print(f"Number of edges in the graph: {new_edge_index.shape[1]}")
print("Missing proteins saved to missing_proteins.csv")

print(f"New index-to-protein mapping: {new_idx_to_protein}")


Total proteins: 1406
Proteins without edges: 0
Proteins with edges: 1406
Number of edges in the graph: 171832
Missing proteins saved to missing_proteins.csv
New index-to-protein mapping: {0: 'MAPK1', 1: 'SUGT1', 2: 'RBX1', 3: 'PEPD', 4: 'EEF1G', 5: 'VTA1', 6: 'LSM5', 7: 'PFDN1', 8: 'PSMD5', 9: 'RAN', 10: 'SMIM5', 11: 'LGALS1', 12: 'SLC35A4', 13: 'CAPZA1', 14: 'PPP1CB', 15: 'NAP1L4', 16: 'CAPZA2', 17: 'CAP1', 18: 'GAS2L1', 19: 'CALM1', 20: 'APRT', 21: 'RPS12', 22: 'CLIC1', 23: 'USP15', 24: 'ABI1', 25: 'DYNLRB1', 26: 'SNAPIN', 27: 'TPM3', 28: 'PSMF1', 29: 'BRK1', 30: 'LGALSL', 31: 'DBN1', 32: 'PFN1', 33: 'TWF2', 34: 'STK24', 35: 'CLIC4', 36: 'LIMS1', 37: 'GLRX', 38: 'MYH9', 39: 'VCP', 40: 'CFL1', 41: 'DSTN', 42: 'ANXA3', 43: 'CA1', 44: 'ATP6V1G1', 45: 'KRT6A', 46: 'RAB14', 47: 'EEF1D', 48: 'PIP4K2A', 49: 'FABP5', 50: 'MAPRE2', 51: 'MAPRE1', 52: 'MYL6', 53: 'MTPN', 54: 'ACTB', 55: 'ACTG1', 56: 'YWHAH', 57: 'CAVIN2', 58: 'RAB11B', 59: 'NRGN', 60: 'ILK', 61: 'YWHAE', 62: 'MTURN', 63: 'YWHAZ

Check graph connectivity - fully connected is better ideally

In [87]:
import torch
import pandas as pd
import networkx as nx
from torch_geometric.utils import to_networkx

try:
    G = to_networkx(edge_index, edge_attr=edge_attr, node_attrs=None)
    print("Graph successfully converted to NetworkX.")
except Exception as e:
    print(f"Error converting to NetworkX: {e}")
    G = None

if G is None:
    print("Manually creating NetworkX graph due to conversion failure.")
    G = nx.Graph() 

    for idx, (u, v) in enumerate(zip(edge_index[0], edge_index[1])):
        G.add_edge(u.item(), v.item(), weight=edge_attr[idx].item())

connected_components_list = list(nx.connected_components(G))
num_components = len(connected_components_list)

largest_component = max(connected_components_list, key=len)
largest_component_size = len(largest_component)

subgraph = G.subgraph(largest_component).copy()

node_mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(largest_component)}

new_edge_index = []
new_edge_attr = []

for u, v, data in subgraph.edges(data=True):
    new_edge_index.append([node_mapping[u], node_mapping[v]]) 
    new_edge_attr.append(data['weight']) 

new_edge_index = torch.tensor(new_edge_index, dtype=torch.long).t().contiguous()
new_edge_attr = torch.tensor(new_edge_attr, dtype=torch.float)

new_protein_to_idx = {new_idx: old_idx for new_idx, old_idx in enumerate(largest_component)}

node_idx_mapping_df = pd.DataFrame(list(new_protein_to_idx.items()), columns=['New_Node_Index', 'Original_Protein_ID'])

node_idx_mapping_df.to_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/node_index_mapping_RF_QRILC.csv', index=False)

edge_index_df = pd.DataFrame(new_edge_index.numpy().T, columns=['Source_Node', 'Target_Node'])
edge_index_df['Edge_Weight'] = new_edge_attr.numpy()

edge_attr_df = pd.DataFrame(new_edge_attr.numpy(), columns=['Edge_Weight'])

edge_index_df.to_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/final_edge_index_RF_QRILC.csv', index=False)
edge_attr_df.to_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/final_edge_attr_RF_QRILC.csv', index=False)

print(f"Final edge_index shape: {new_edge_index.shape}")
print(f"Final edge_attr shape: {new_edge_attr.shape}")
print(f"Node index mapping saved to: node_index_mapping.csv")
print(f"Edge index saved to: edge_index.csv")
print(f"Edge attributes saved to: edge_attr.csv")


Error converting to NetworkX: to_networkx() got an unexpected keyword argument 'edge_attr'
Manually creating NetworkX graph due to conversion failure.
Final edge_index shape: torch.Size([2, 85916])
Final edge_attr shape: torch.Size([85916])
Node index mapping saved to: node_index_mapping.csv
Edge index saved to: edge_index.csv
Edge attributes saved to: edge_attr.csv


In [88]:
import torch
import pandas as pd
from torch_geometric.data import Data

edge_index_df = pd.read_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/final_edge_index_RF_QRILC.csv')

edge_index = edge_index_df[['Source_Node', 'Target_Node']].values.T  
edge_index = torch.tensor(edge_index, dtype=torch.long)


edge_attr = edge_index_df['Edge_Weight'].values  
edge_attr = torch.tensor(edge_attr, dtype=torch.float)

num_nodes = edge_index.max().item() + 1  
node_features = torch.zeros((num_nodes, 1))

edge_index, edge_attr = pyg_utils.to_undirected(edge_index, edge_attr=edge_attr)

data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr)

print(data)


Data(x=[1406, 1], edge_index=[2, 171832], edge_attr=[171832])


In [89]:
data.is_undirected()

True

In [ ]:
import pandas as pd

df_original = pd.read_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/node_index_mapping_RF_QRILC.csv')  
df_gene_mapping = pd.read_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/original_node_indices_RF_QRILC.csv') 

merged_df = pd.merge(df_original, df_gene_mapping[['node_index', 'Gene']], how='left', 
                     left_on='Original_Protein_ID', right_on='node_index')

merged_df = merged_df.drop(columns=['node_index'])

merged_df.to_csv('./PlasmaAAA/PreprocessingScripts/CohortSpecificImpute/finalnode_info_with_Genes_RF_QRILC.csv', index=False)

print(merged_df.head(10))


   New_Node_Index  Original_Protein_ID   Gene
0               0                    0  MAPK1
1               1                    1  SUGT1
2               2                    2   RBX1
3               3                    3   PEPD
4               4                    4  EEF1G
